In [ ]:

try:
    import kaggle_benchmarks as kbench
except ImportError:
    # Failsafe: Mock the kbench harness if the library is not found (e.g. during build/commit)
    import types
    class MockTask:
        def __init__(self, f): self.f = f
        def run(self, *args, **kwargs): return self.f(*args, **kwargs)
        def evaluate(self, *args, **kwargs):
            class Res: 
                def as_dataframe(self): return None
            return Res()
        def __call__(self, *args, **kwargs): return self.f(*args, **kwargs)
    kbench = types.SimpleNamespace()
    kbench.task = lambda **kwargs: lambda f: MockTask(f)
    kbench.llm = types.SimpleNamespace(prompt=lambda p: '{"final_answer": "0.0"}')
    print('⚠️ kaggle_benchmarks not found. Running in Failsafe (Mock) mode.')

import json
import re
import math
from datetime import datetime

# (Rest of the utils remain the same...)
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

# Task 13: Scattering Amplitude Constant
TASK_ID = "fp_13"
GROUND_TRUTH = -0.167

@kbench.task(name="FP-13 Scattering Amplitude Constant", description="Physics")
def task_13(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nWe examine the high-energy limit (sqrt(s) >> M_W) of the scattering process d_R + dbar_L -> W_L- + W_L+. In the simplified electroweak theory provided: Vertices: The photon couples to quarks with V_gamma_qq = +i(e/3)gamma^mu and to W-bosons with the standard Yang-Mills vertex. Helicity: The incoming d_R state forbids the t-channel diagram, isolating the s-channel photon diagram. Polarization: The outgoing longitudinal W-bosons are approximated by epsilon_L^mu(k) approx k^mu / M_W. As s -> infinity, the leading behavior of the total scattering amplitude M_total can be written in the form: M_total approx i * K * (e^2 / M_W^2 * s * sin(theta)) where K is a dimensionless real number. What is the exact value of K? Give your answer as a decimal rounded to three decimal places.\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_13.run(kbench.llm)
